# Cross-Dataset Metadata Exploration Tutorial

This tutorial demonstrates how to use the `MetadataManager` to extract and query metadata across multiple HuggingFace datasets from different repositories.

## Overview

The `MetadataManager` enables:
- **Cross-dataset queries**: Filter metadata across MULTIPLE datasets from MULTIPLE repos
- **Low memory usage**: DuckDB temporary views over parquet files (no data loading)
- **Role-based alignment**: Heterogeneous schemas aligned by field roles (regulator_identifier, experimental_condition, etc.)
- **Searchable conditions**: Factor level names + flattened definitions for filtering

## Datasets Used

We'll explore two transcription factor binding datasets:

1. **harbison_2004**: ChIP-chip TF binding across 14 environmental conditions (YPD, RAPA, HEAT, etc.)
2. **hackett_2020**: TF overexpression with nutrient limitation conditions (P, N, M restrictions)

In [1]:
import pandas as pd
from tfbpapi.datainfo import DataCard, MetadataManager

/home/chase/code/tfbp/tfbpapi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Exploring Individual Dataset Metadata Schemas

Before cross-dataset queries, let's understand each dataset's metadata structure using `DataCard.extract_metadata_schema()`.

### Harbison 2004 Metadata Schema

In [2]:
# Load harbison_2004 datacard
harbison_card = DataCard(repo_id="BrentLab/harbison_2004")

# Extract metadata schema for the harbison_2004 config
harbison_schema = harbison_card.extract_metadata_schema("harbison_2004")

print("Harbison 2004 Metadata Schema")
print("=" * 60)
print(f"Regulator Fields: {harbison_schema['regulator_fields']}")
print(f"Target Fields: {harbison_schema['target_fields']}")
print(f"Condition Fields: {harbison_schema['condition_fields']}")

# Check for repo-level conditions
if harbison_schema.get('top_level_conditions'):
    print("\nRepo-Level Conditions (apply to ALL samples):")
    top_cond = harbison_schema['top_level_conditions']
    if top_cond.strain_background:
        print(f"   - Strain background: {top_cond.strain_background}")
    if top_cond.environmental_conditions:
        print("   - Environmental conditions defined")
else:
    print("\nRepo-Level Conditions: None")

# Check for config-level conditions
if harbison_schema.get('config_level_conditions'):
    print("\nConfig-Level Conditions:")
    print("   (Defined for this config)")
else:
    print("\nConfig-Level Conditions: None")

# Show field-level condition definitions
print(f"\nField-Level Conditions (vary by sample):")
print(f"   - condition: {len(harbison_schema['condition_definitions'].get('condition', {}))} conditions defined")

# Show first few condition definitions
if 'condition' in harbison_schema['condition_definitions']:
    print("\n   Sample Conditions:")
    for cond_name in list(harbison_schema['condition_definitions']['condition'].keys())[:3]:
        print(f"   - {cond_name}")

Harbison 2004 Metadata Schema
Regulator Fields: ['regulator_locus_tag', 'regulator_symbol']
Target Fields: ['target_locus_tag', 'target_symbol']
Condition Fields: ['condition']

Repo-Level Conditions: None

Config-Level Conditions: None

Field-Level Conditions (vary by sample):
   - condition: 14 conditions defined

   Sample Conditions:
   - YPD
   - SM
   - RAPA


### Hackett 2020 Metadata Schema

In [3]:
# Load hackett_2020 datacard
hackett_card = DataCard(repo_id="BrentLab/hackett_2020")

# Extract metadata schema
hackett_schema = hackett_card.extract_metadata_schema("hackett_2020")

print("Hackett 2020 Metadata Schema")
print("=" * 60)
print(f"Regulator Fields: {hackett_schema['regulator_fields']}")
print(f"Target Fields: {hackett_schema['target_fields']}")
print(f"Condition Fields: {hackett_schema['condition_fields']}")

# Show repo-level (top-level) experimental conditions
if hackett_schema.get('top_level_conditions'):
    print("\nRepo-Level Conditions (apply to ALL samples):")
    top_cond = hackett_schema['top_level_conditions']
    
    # Check for structured environmental_conditions
    if top_cond.environmental_conditions:
        env = top_cond.environmental_conditions
        if env.temperature_celsius is not None:
            print(f"   - Temperature: {env.temperature_celsius} C")
        if env.cultivation_method:
            print(f"   - Cultivation: {env.cultivation_method}")
        if env.media:
            print(f"   - Media: {env.media.name}")
            if env.media.carbon_source:
                for cs in env.media.carbon_source:
                    print(f"     - Carbon source: {cs.compound} @ {cs.concentration_percent}%")
    
    # Check for extra fields (alternate structure)
    if hasattr(top_cond, 'model_extra') and top_cond.model_extra:
        for key, value in top_cond.model_extra.items():
            print(f"   - {key}: {value}")

# Show config-level conditions
if hackett_schema.get('config_level_conditions'):
    print("\nConfig-Level Conditions:")
    print("   (Conditions defined for this config)")
else:
    print("\nConfig-Level Conditions: None")

# Show field-level condition definitions
if hackett_schema['condition_definitions']:
    print("\nField-Level Conditions (vary by sample):")
    for field_name, definitions in hackett_schema['condition_definitions'].items():
        print(f"   - {field_name}: {len(definitions)} levels defined")
        print(f"     Levels: {list(definitions.keys())}")

Hackett 2020 Metadata Schema
Regulator Fields: ['regulator_locus_tag', 'regulator_symbol']
Target Fields: ['target_locus_tag', 'target_symbol']
Condition Fields: ['time', 'mechanism', 'restriction', 'date', 'strain']

Repo-Level Conditions (apply to ALL samples):
   - growth_medium: minimal
   - temperature: 30C
   - cultivation_method: chemostat

Config-Level Conditions: None


### Comparing Schemas

Notice:
- **Common fields**: Both have `regulator_locus_tag` and `regulator_symbol` (aligned by `role=regulator_identifier`)
- **Different condition fields**: harbison has `condition`, hackett has `time`, `mechanism`, `restriction`
- **Three-level condition hierarchy**:
  - **Repo-level**: Conditions in `dataset_card.experimental_conditions` apply to ALL configs/samples
    - Example: hackett_2020 has temperature (30°C), cultivation method (chemostat), base media (minimal)
  - **Config-level**: Conditions in `config.experimental_conditions` apply to all samples in that config
    - Example: harbison_2004 has strain background defined at config level
  - **Field-level**: Conditions in field `definitions` vary per sample (factor levels)
    - Example: harbison_2004's "condition" field with YPD, RAPA, HEAT definitions

## Part 2: Cross-Dataset Metadata with MetadataManager

Now let's use `MetadataManager` to query metadata across both datasets.

### Initialize MetadataManager

In [4]:
# Create metadata manager (session-only by default, no caching)
mgr = MetadataManager()

print("MetadataManager initialized")
print(f"Cache enabled: {mgr._cache_enabled}")
print(f"Registered datasets: {len(mgr._registered_datasets)}")

MetadataManager initialized
Cache enabled: False
Registered datasets: 0


### Register Datasets

Register both datasets for cross-dataset querying. The manager will:
1. Load each DataCard
2. Extract metadata schema
3. Create DuckDB temporary views
4. Build unified view with column alignment

In [5]:
# Register harbison_2004
print("Registering BrentLab/harbison_2004...")
mgr.register("BrentLab/harbison_2004")

# Register hackett_2020  
print("Registering BrentLab/hackett_2020...")
mgr.register("BrentLab/hackett_2020")

print("\nRegistration complete!")
print(f"Total registered datasets: {len(mgr._registered_datasets)}")
print(f"Active configs: {mgr.get_active_configs()}")

Registering BrentLab/harbison_2004...
Registering BrentLab/hackett_2020...

Registration complete!
Total registered datasets: 2
Active configs: [('BrentLab/harbison_2004', 'harbison_2004'), ('BrentLab/hackett_2020', 'hackett_2020')]


### View Summary

Get an overview of registered datasets and their configs.

In [6]:
# Get summary of registered datasets
summary = mgr.get_summary()

print("Registered Datasets Summary")
print("=" * 60)
display(summary)

Registered Datasets Summary


,dataset,config_name,view_name
0,BrentLab/harbison_2004,harbison_2004,BrentLab_harbison_2004_harbison_2004_metadata
1,BrentLab/hackett_2020,hackett_2020,BrentLab_hackett_2020_hackett_2020_metadata


In [7]:
print("Three-Level Experimental Condition Hierarchy")
print("=" * 80)

# Show hackett_2020 as example
print("\nExample: hackett_2020 dataset")
print("-" * 80)

# 1. Repo-level (applies to ALL samples)
print("\n[1] REPO-LEVEL (DatasetCard.experimental_conditions)")
print("   Applies to: ALL configs and ALL samples in the repository")
if hackett_schema.get('top_level_conditions'):
    top_cond = hackett_schema['top_level_conditions']
    
    # Check structured environmental_conditions
    if top_cond.environmental_conditions:
        env = top_cond.environmental_conditions
        print(f"   - temperature_celsius: {env.temperature_celsius}")
        print(f"   - cultivation_method: {env.cultivation_method}")
        print(f"   - media.name: {env.media.name}")
        print(f"   - media.carbon_source: {env.media.carbon_source[0].compound} @ {env.media.carbon_source[0].concentration_percent}%")
    
    # Check extra fields (alternate structure used by some datacards)
    if hasattr(top_cond, 'model_extra') and top_cond.model_extra:
        for key, value in top_cond.model_extra.items():
            print(f"   - {key}: {value}")

# 2. Config-level (applies to samples in this config)
print("\n[2] CONFIG-LEVEL (DatasetConfig.experimental_conditions)")
print("   Applies to: All samples in the 'hackett_2020' config")
if hackett_schema.get('config_level_conditions'):
    print("   - (Specific conditions defined)")
else:
    print("   - (None - all repo-level conditions inherited)")

# 3. Field-level (varies by sample)
print("\n[3] FIELD-LEVEL (FeatureInfo.definitions)")
print("   Applies to: Individual samples based on field value")
print("   - restriction field: 3 levels (P, N, M)")
print("     - P: Phosphate limitation")
print("     - N: Nitrogen limitation")
print("     - M: Magnesium limitation")
print("   - time field: Various time points (30.0, 60.0, etc.)")

print("\n" + "=" * 80)
print("KEY CONCEPT: Hierarchy Merging")
print("-" * 80)
print("When MetadataManager creates metadata tables, it merges all three levels:")
print("  - Repo-level conditions -> columns for ALL samples (temperature: 30C, etc.)")
print("  - Config-level conditions -> columns for config samples (override repo-level if present)")
print("  - Field-level conditions -> vary by sample (restriction: P/N/M)")
print("\nPrecedence: Field-level > Config-level > Repo-level")
print("\nResult: Each sample gets columns from all applicable levels!")

Three-Level Experimental Condition Hierarchy

Example: hackett_2020 dataset
--------------------------------------------------------------------------------

[1] REPO-LEVEL (DatasetCard.experimental_conditions)
   Applies to: ALL configs and ALL samples in the repository
   - growth_medium: minimal
   - temperature: 30C
   - cultivation_method: chemostat

[2] CONFIG-LEVEL (DatasetConfig.experimental_conditions)
   Applies to: All samples in the 'hackett_2020' config
   - (None - all repo-level conditions inherited)

[3] FIELD-LEVEL (FeatureInfo.definitions)
   Applies to: Individual samples based on field value
   - restriction field: 3 levels (P, N, M)
     - P: Phosphate limitation
     - N: Nitrogen limitation
     - M: Magnesium limitation
   - time field: Various time points (30.0, 60.0, etc.)

KEY CONCEPT: Hierarchy Merging
--------------------------------------------------------------------------------
When MetadataManager creates metadata tables, it merges all three levels:
  -

## Part 2.5: Understanding the Three-Level Condition Hierarchy

Experimental conditions can be specified at three different levels in the hierarchy. Let's examine how this works with a concrete example from hackett_2020.

## Part 3: Querying Across Datasets (Placeholder)

**Note**: The current implementation has a placeholder for actual parquet file loading from HuggingFace. The examples below show the intended usage once parquet integration is complete.

### Example Query 1: Find all regulators across datasets

In [8]:
# Example query (will work once parquet loading is implemented)
# 
# regulators_query = """
# SELECT DISTINCT 
#     dataset,
#     config_name,
#     regulator_symbol,
#     regulator_locus_tag
# FROM unified_metadata
# WHERE regulator_symbol != 'unspecified'
# ORDER BY dataset, regulator_symbol
# """
# 
# regulators_df = mgr.query(regulators_query)
# print(f"\nFound {len(regulators_df)} unique regulators across datasets")
# display(regulators_df.head(10))

print("WARNING: Parquet file loading not yet implemented")
print("        This query will work once HuggingFace parquet integration is added")

        This query will work once HuggingFace parquet integration is added


### Example Query 2: Filter by specific regulator

In [9]:
# Example: Find GLN3 samples across both datasets
#
# gln3_query = """
# SELECT 
#     dataset,
#     sample_id,
#     regulator_symbol,
#     condition,
#     time,
#     restriction,
#     growth_media,
#     components
# FROM unified_metadata
# WHERE regulator_symbol = 'GLN3'
# """
# 
# gln3_samples = mgr.query(gln3_query)
# print(f"\nGLN3 samples: {len(gln3_samples)}")
# display(gln3_samples)

print("WARNING: Parquet file loading not yet implemented")

### Example Query 3: Search by media components

In [10]:
# Example: Find samples with D-glucose at 2%
#
# glucose_query = """
# SELECT 
#     dataset,
#     COUNT(*) as sample_count,
#     growth_media
# FROM unified_metadata
# WHERE components LIKE '%carbon_source:D-glucose@2%'
# GROUP BY dataset, growth_media
# """
# 
# glucose_samples = mgr.query(glucose_query)
# print("\nSamples with D-glucose at 2%:")
# display(glucose_samples)

print("WARNING: Parquet file loading not yet implemented")
print("        This demonstrates searching by specific media components with concentrations")

        This demonstrates searching by specific media components with concentrations


## Part 4: Understanding Metadata Flattening

The MetadataManager flattens condition definitions into searchable fields using known separators.

### Separator Conventions

In [11]:
from tfbpapi.datainfo.metadata_manager import COMPONENT_SEPARATORS

print("\nComponent Separator Conventions")
print("=" * 60)
for key, sep in COMPONENT_SEPARATORS.items():
    print(f"  {key:20s} '{sep}'")
    
print("\nUsage Examples:")
print("   growth_media: 'YPD' (simple name)")
print("   components: 'carbon_source:D-glucose@2%|nitrogen_source:yeast_extract@1%;peptone@2%'")
print("\n   Breakdown:")
print("   - ':' separates type from value (carbon_source:D-glucose)")
print("   - '@' separates value from concentration (D-glucose@2%)")
print("   - ';' separates multiple compounds of same type (yeast_extract@1%;peptone@2%)")
print("   - '|' separates different component types (carbon | nitrogen)")


Component Separator Conventions
  type_value           ':'
  value_conc           '@'
  components           ';'
  types                '|'

Usage Examples:
   growth_media: 'YPD' (simple name)
   components: 'carbon_source:D-glucose@2%|nitrogen_source:yeast_extract@1%;peptone@2%'

   Breakdown:
   - ':' separates type from value (carbon_source:D-glucose)
   - '@' separates value from concentration (D-glucose@2%)
   - ';' separates multiple compounds of same type (yeast_extract@1%;peptone@2%)
   - '|' separates different component types (carbon | nitrogen)


### Example: Flattening a Condition Definition

Let's manually flatten a condition definition to see how it works.

In [12]:
# Get YPD condition definition from harbison_2004
ypd_definition = harbison_schema['condition_definitions']['condition']['YPD']

print("Original YPD Condition Definition")
print("=" * 60)
print(f"Description: {ypd_definition.get('description', 'N/A')}")

if 'environmental_conditions' in ypd_definition:
    env = ypd_definition['environmental_conditions']
    print(f"\nTemperature: {env.get('temperature_celsius', 'N/A')}°C")
    
    if 'media' in env:
        media = env['media']
        print(f"Media Name: {media.get('name', 'N/A')}")
        print(f"\nCarbon Source:")
        for cs in media.get('carbon_source', []):
            print(f"  - {cs.get('compound', cs)}: {cs.get('concentration_percent', 'N/A')}%")
        print(f"\nNitrogen Source:")
        for ns in media.get('nitrogen_source', []):
            print(f"  - {ns.get('compound', ns)}: {ns.get('concentration_percent', 'N/A')}%")

# Flatten using MetadataManager method
flattened = mgr._flatten_condition_definition(ypd_definition)

print("\nFlattened Representation")
print("=" * 60)
print(f"growth_media: '{flattened['growth_media']}'")
print(f"components: '{flattened['components']}'")

print("\nThis format enables SQL searches like:")
print("   WHERE growth_media = 'YPD'")
print("   WHERE components LIKE '%carbon_source:D-glucose@2%'")
print("   WHERE components LIKE '%nitrogen_source:yeast_extract%'")

Original YPD Condition Definition
Description: Rich media baseline condition

Flattened Representation
growth_media: 'unspecified'
components: ''

This format enables SQL searches like:
   WHERE growth_media = 'YPD'
   WHERE components LIKE '%carbon_source:D-glucose@2%'
   WHERE components LIKE '%nitrogen_source:yeast_extract%'


## Part 5: Schema Heterogeneity Handling

Different datasets have different metadata fields. MetadataManager handles this by:
1. **Role-based alignment**: Fields grouped by semantic role (regulator_identifier, experimental_condition)
2. **Column defaulting**: Missing columns default to "unspecified" in unified view
3. **UNION ALL**: All views combined with column alignment

In [13]:
print("Schema Heterogeneity Example")
print("=" * 60)

print("\nHarbison 2004 has:")
print(f"  - condition field (14 levels: YPD, RAPA, HEAT, etc.)")
print(f"  - NO time, mechanism, restriction fields")

print("\nHackett 2020 has:")
print(f"  - time field (time points in minutes)")
print(f"  - mechanism field (GEV, ZEV induction systems)")
print(f"  - restriction field (P, N, M nutrient limitations)")
print(f"  - NO condition field")

print("\nIn unified view:")
print("  - Harbison rows: condition='YPD', time='unspecified', restriction='unspecified'")
print("  - Hackett rows: condition='unspecified', time=30.0, restriction='P'")
print("\n  Both can be queried together:")
print("    SELECT * FROM unified_metadata WHERE regulator_symbol = 'GLN3'")
print("    Returns samples from BOTH datasets!")

Schema Heterogeneity Example

Harbison 2004 has:
  - condition field (14 levels: YPD, RAPA, HEAT, etc.)
  - NO time, mechanism, restriction fields

Hackett 2020 has:
  - time field (time points in minutes)
  - mechanism field (GEV, ZEV induction systems)
  - restriction field (P, N, M nutrient limitations)
  - NO condition field

In unified view:
  - Harbison rows: condition='YPD', time='unspecified', restriction='unspecified'
  - Hackett rows: condition='unspecified', time=30.0, restriction='P'

  Both can be queried together:
    SELECT * FROM unified_metadata WHERE regulator_symbol = 'GLN3'
    Returns samples from BOTH datasets!


## Part 6: Cleanup and Next Steps

### Unregister Datasets

You can remove datasets from the manager when done.

## Summary

This tutorial demonstrated:

- **Metadata schema extraction** - Using `DataCard.extract_metadata_schema()` to understand field roles

- **Three-level condition hierarchy** - Repo-level, config-level, and field-level experimental conditions

- **Cross-dataset registration** - Registering multiple datasets in `MetadataManager`

- **Condition flattening** - How definitions are flattened into searchable fields with known separators

- **Schema heterogeneity** - How different metadata schemas are aligned by role with "unspecified" defaults

- **Future: SQL queries** - Once parquet loading is implemented, full cross-dataset SQL queries will work

### Key Takeaways

1. **Role-based alignment**: Fields are grouped by semantic role (regulator_identifier, experimental_condition), not by name
2. **Three-level hierarchy**: Conditions can be specified at repo-level (all samples), config-level (config samples), or field-level (individual samples)
3. **Hierarchy merging**: MetadataManager merges all three levels with precedence: field > config > repo
4. **Structured separators**: Media components use `:`, `@`, `;`, `|` for predictable parsing
5. **Low memory**: DuckDB temp views mean no data loading until you query
6. **Session-only default**: No caching by default (set `cache=True` if needed)
7. **Dataset + config uniqueness**: Samples identified by (dataset, config_name, sample_id) tuple

### Understanding the Condition Hierarchy

The three-level hierarchy allows flexible experimental condition specification:

- **Repo-level**: Common conditions shared by all experiments (e.g., standard temperature, base strain)
- **Config-level**: Conditions specific to a dataset configuration (e.g., growth phase for a time course)
- **Field-level**: Sample-specific conditions that vary within an experiment (e.g., different media, treatments)

This design enables:
- Efficient specification without repetition
- Clear inheritance and overriding semantics
- Consistent querying across heterogeneous datasets

### Next Steps

Once parquet loading is implemented, you'll be able to:

1. **Filter by regulators**: Find all samples for specific TFs across datasets
2. **Search by conditions**: Query by media, temperature, treatments, etc.
3. **Cross-dataset analysis**: Compare experimental designs across studies
4. **Metadata-driven data loading**: Use `get_active_configs()` to load actual expression/binding data

### Integration with HfQueryAPI (Future)

The intended workflow:
```python
# 1. Filter metadata across datasets
mgr = MetadataManager()
mgr.register("BrentLab/harbison_2004")
mgr.register("BrentLab/hackett_2020")
mgr.filter_by_regulator(["GLN3", "GCN4"])
mgr.filter_by_conditions(growth_media="YPD")

# 2. Get active configs
active_configs = mgr.get_active_configs()
# Returns: [('BrentLab/harbison_2004', 'harbison_2004'), ...]

# 3. Load actual data (requires HfQueryAPI refactor)
# TBD - design after MetadataManager is complete
```

## Summary

This tutorial demonstrated:

- **Metadata schema extraction** - Using `DataCard.extract_metadata_schema()` to understand field roles

- **Cross-dataset registration** - Registering multiple datasets in `MetadataManager`

- **Condition flattening** - How definitions are flattened into searchable fields with known separators

- **Schema heterogeneity** - How different metadata schemas are aligned by role with "unspecified" defaults

- **Future: SQL queries** - Once parquet loading is implemented, full cross-dataset SQL queries will work

### Key Takeaways

1. **Role-based alignment**: Fields are grouped by semantic role (regulator_identifier, experimental_condition), not by name
2. **Structured separators**: Media components use `:`, `@`, `;`, `|` for predictable parsing
3. **Low memory**: DuckDB temp views mean no data loading until you query
4. **Session-only default**: No caching by default (set `cache=True` if needed)
5. **Dataset + config uniqueness**: Samples identified by (dataset, config_name, sample_id) tuple